In [ ]:
!pip install openmm pdbfixer


In [ ]:
import os
import re
import time
import json
import torch
import numpy as np
from pdbfixer import PDBFixer
from openmm.app import *
from openmm import *
from openmm.unit import *
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2.credentials import Credentials

print("==================================================")
print("  FOLDPIPE REAL PHYSICS PRODUCTION PIPELINE (MD)  ")
print("==================================================\n")

# ------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------
DRIVE_FOLDER_ID = "1Few5wzRuuhlwbj4DJD9nkOP98t_QqZcz"
TOTAL_FRAMES = 1_000_000
FRAMES_PER_CHUNK = 10000  # Frames per checkpoint file (16 mins per upload)
TOTAL_CHUNKS = TOTAL_FRAMES // FRAMES_PER_CHUNK
STEPS_PER_FRAME = 100  # 100 MD steps between saved frames (0.2 picoseconds per frame)
MAX_UPLOAD_RETRIES = 3

# ------------------------------------------------------------------
# STEP 1: AUTHENTICATE GOOGLE DRIVE
# ------------------------------------------------------------------
print("[1/5] Authenticating Google Drive via OAuth...")
import glob
secret_path = "/kaggle/input/gcp-secret-dataset/token.json"
possible_paths = glob.glob('/kaggle/input/**/token.json', recursive=True)
if possible_paths: secret_path = possible_paths[0]

with open(secret_path, 'r') as f:
    creds_json = json.load(f)
credentials = Credentials.from_authorized_user_info(
    creds_json, scopes=['https://www.googleapis.com/auth/drive']
)
drive_service = build('drive', 'v3', credentials=credentials)
print("  ✔ Authenticated successfully.")

# ------------------------------------------------------------------
# STEP 2: RESTARTABILITY (CHECK GOOGLE DRIVE)
# ------------------------------------------------------------------
print("[2/5] Checking Google Drive for existing checkpoints...")
start_chunk = 0
try:
    query = f"'{DRIVE_FOLDER_ID}' in parents and name contains 'checkpoint_batch_' and trashed = false"
    results = drive_service.files().list(q=query, fields="files(name)").execute()
    files = results.get('files', [])
    
    existing = []
    for f in files:
        match = re.search(r'checkpoint_batch_(\d+)\.pt', f['name'])
        if match:
            existing.append(int(match.group(1)))
            
    if existing:
        last_completed = max(existing)
        start_chunk = last_completed + 1
        print(f"  ✔ RESUMING: Found checkpoint {last_completed}. Starting at chunk {start_chunk}/{TOTAL_CHUNKS}.")
    else:
        print("  ✔ No prior checkpoints found. Starting fresh run.")
except Exception as e:
    print(f"  ⚠️ Warning: Could not query Drive ({e}). Starting from chunk 0.")

if start_chunk >= TOTAL_CHUNKS:
    print("\n🎉 Simulation already complete!")
    import sys
    sys.exit(0)

# ------------------------------------------------------------------
# STEP 3: AUTOMATED PDB PREPARATION & FIXING
# ------------------------------------------------------------------
print("[3/5] Fetching and repairing Human Prion (1QLX) topology...")
fixer = PDBFixer(pdbid='1QLX')
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.0)  # Physiological pH 7.0
print("  ✔ PDB structure fixed and hydrogens added.")

# ------------------------------------------------------------------
# STEP 4: OPENMM SYSTEM INITIALIZATION & MINIMIZATION
# ------------------------------------------------------------------
print("[4/5] Setting up OpenMM ForceField and Energy Minimization...")
forcefield = ForceField('amber14-all.xml', 'implicit/obc2.xml')
system = forcefield.createSystem(fixer.topology, nonbondedMethod=NoCutoff, constraints=HBonds)
integrator = LangevinIntegrator(300 * kelvin, 1 / picosecond, 0.002 * picoseconds)

try:
    platform = Platform.getPlatformByName('CUDA')
except Exception:
    print("  ⚠️ CUDA not registered. Falling back to OpenCL (GPU Hardware)...")
    platform = Platform.getPlatformByName('OpenCL')

simulation = Simulation(fixer.topology, system, integrator, platform)
simulation.context.setPositions(fixer.positions)

print("  Running Energy Minimization (preventing atomic explosion)...")
simulation.minimizeEnergy()
print("  ✔ System minimized and stable.")

# ------------------------------------------------------------------
# STEP 5: THE REAL PHYSICS EXECUTION & STREAMING LOOP
# ------------------------------------------------------------------
print("[5/5] Starting Real MD Simulation Loop...")
print("=" * 70)

os.makedirs('/kaggle/working/prion_ensemble', exist_ok=True)
sim_start_time = time.time()

for chunk_idx in range(start_chunk, TOTAL_CHUNKS):
    chunk_buffer = []
    chunk_start_time = time.time()
    
    # Generate 64 frames per chunk
    for _ in range(FRAMES_PER_CHUNK):
        simulation.step(STEPS_PER_FRAME)
        state = simulation.context.getState(getPositions=True)
        positions = state.getPositions(asNumpy=True).value_in_unit(nanometer)
        
        # Convert to Torch Tensor (FP32)
        tensor_pos = torch.tensor(positions, dtype=torch.float32)
        chunk_buffer.append(tensor_pos)
        
    # Stack into shape: [10000, Num_Atoms, 3]
    chunk_tensor = torch.stack(chunk_buffer, dim=0)
    
    file_name = f"checkpoint_batch_{chunk_idx}.pt"
    local_path = f"/kaggle/working/prion_ensemble/{file_name}"
    torch.save(chunk_tensor, local_path)
    
    # Upload with Retry Logic
    upload_success = False
    for attempt in range(1, MAX_UPLOAD_RETRIES + 1):
        try:
            file_metadata = {'name': file_name, 'parents': [DRIVE_FOLDER_ID]}
            media = MediaFileUpload(local_path, resumable=True)
            drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
            upload_success = True
            break
        except Exception as e:
            time.sleep(5)
            
    if not upload_success:
        raise RuntimeError(f"CRITICAL: Upload failed for chunk {chunk_idx} after {MAX_UPLOAD_RETRIES} attempts.")
        
    os.remove(local_path)
    
    elapsed = time.time() - chunk_start_time
    print(f"[{time.strftime('%H:%M:%S')}] Chunk {chunk_idx}/{TOTAL_CHUNKS} "
          f"({(chunk_idx/TOTAL_CHUNKS)*100:.1f}%) | Time: {elapsed:.1f}s | Upload: OK")

print("\n" + "=" * 70)
print("\U0001f389 REAL MD SIMULATION COMPLETED AND STREAMED TO DRIVE!")
print("==================================================")
